# Article Link Replacements

Create a lightweight agent which is shown the headline, old article URL and rough date (just use the edition date) to web search and try to find alternative live articles covering the same story

In [11]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI
from pydantic import BaseModel
import requests
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool
data_path = "../data"

load_dotenv(override=True)
openAI_api_key = os.getenv('OPENAI_API_KEY')
giphy_api_key = os.getenv("GIPHY_API_KEY")

MODEL = "gpt-5.4-mini"

if not openAI_api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not openAI_api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif openAI_api_key.strip() != openAI_api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

client = OpenAI()

API key found and looks good so far!


Testing the syntax

In [ ]:
agent = Agent(name="Test", instructions="You confirm that I know how to use OpenAI Agents SDK", model="gpt-5.4-mini")
result = await Runner.run(agent, "Has this worked?")
print(result.final_output)

Yes — it worked.


Adding a web search tool

In [13]:
@function_tool
def search_web(query: str) -> str:
    """Search the web using Serper.

    Args:
        query: The search query to send to the web search engine.
    """
    response = requests.post(
        "https://google.serper.dev/search",
        headers={
            "X-API-KEY": os.environ["SERPER_API_KEY"],
            "Content-Type": "application/json",
        },
        json={
            "q": query,
        },
        timeout=10,
    )

    response.raise_for_status()

    results = response.json()

    return str(results)

In [14]:
searcher = Agent(name="Searcher", 
                 model="gpt-5.4-mini", 
                 instructions="You perform a web search based on provided inputs and synthesise an answer based on what the web search returns", 
                 tools=[search_web])

In [15]:
search = await Runner.run(searcher, "What's happening in the bond markets?")

print(search.final_output)

Bond markets are still in a **higher-yield, steeper-curve** regime.

- **U.S. 10-year Treasury** is around **4.7%**, near the upper end of its recent range.
- The big theme is a **steeper yield curve**: longer-term yields are staying elevated relative to short-term rates.
- That usually reflects a mix of **sticky inflation**, **fiscal supply**, and markets expecting **less central-bank support** than in the low-rate era.
- High yields are making **fixed income more attractive again**, but they also keep pressure on **duration** and other rate-sensitive assets.

In short: **bonds are offering more income, but price volatility remains high because rates are still elevated.**

If you want, I can break this down by **Treasuries, corporates, or mortgage bonds**.


Refining for the task

In [32]:
class SearchResult(BaseModel):
    title: str
    url: str


class SearchResults(BaseModel):
    results: list[SearchResult]

In [37]:
replacer = Agent(
    name="Replacer", 
    model="gpt-5.4-mini", 
    instructions="""
    I have URLs to web news articles which no longer work (e.g. return error 404, 460, connection times out).
    I will provide you with:
    -The approximate date for when this article was released
    -A headline related to the article
    -The dead URL
    With this information I want you to search the internet for an alternative article which covers the same story.
    You are provided with the dead URL that I have so that you do not return that to me.
    You must check that the URL you provide returns a live web page.
    Return up to three live alternative URLs along with article titles covering the story.
    """, 
    tools=[search_web],
    output_type=SearchResults
)

In [38]:
replace = await Runner.run(replacer, "2020-05-16, Italian designer creates the 'trikini,' a bikini set with matching face mask, https://alt1037dfw.radio.com/blogs/camfield/designer-creates-the-trikini-a-bikini-with-matching-face")

In [36]:
replace

RunResult(input="Italian designer creates the 'trikini,' a bikini set with matching face mask, 2020-05-16", new_items=[ToolCallItem(agent=Agent(name='Replacer', handoff_description=None, tools=[FunctionTool(name='search_web', description='Search the web using Serper.', params_json_schema={'properties': {'query': {'description': 'The search query to send to the web search engine.', 'title': 'Query', 'type': 'string'}}, 'required': ['query'], 'title': 'search_web_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x0000028944061DC0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None, allowed_callers=None, output_json_schema=None)], mcp_servers=[], mcp_config={}, instructions='\n    I have URLs to web news a

In [39]:
for result in replace.final_output.results:
    print(result.title)
    print(result.url)

Reuters: Italian designer creates 'trikini' bikini with matching face mask
https://www.reuters.com/article/us-health-coronavirus-fashion-idUSKBN22T0C2
CNN: Italian designer creates a bikini with matching face mask amid coronavirus outbreak
https://www.cnn.com/style/article/italian-designer-trikini-bikini-face-mask/index.html
The Guardian: Italian designer creates 'trikini' bikini with matching face mask
https://www.theguardian.com/fashion/2020/may/16/italian-designer-creates-trikini-bikini-with-matching-face-mask


Full workflow

In [48]:
with open(f"{data_path}/known_issues_outstanding.json", "r") as f:
    issues = json.load(f)

issue_types = ["connection_timeout", "460", "404"]
article_replacements = {}

for date in issues:
    if issues[date]["headline_issues"]:
        for headline_issue in issues[date]["headline_issues"]:
            if headline_issue["type"] in issue_types:
                headline = (headline_issue["headline"])
                original_url = (headline_issue["url"])
            article_replacements[date] = {
                    "headline": headline,
                    "original URL": original_url
                }

article_replacements

{'2020-02-01': {'headline': 'Need a loo fast? Book a ‘mobile toilet’ trailer on your smartphone',
  'original URL': 'http://www.asahi.com/ajw/articles/AJ202001260004.html'},
 '2020-05-16': {'headline': "Italian designer creates the 'trikini,' a bikini set with matching face mask",
  'original URL': 'https://alt1037dfw.radio.com/blogs/camfield/designer-creates-the-trikini-a-bikini-with-matching-face'},
 '2020-06-20': {'headline': 'This airport in Taiwan is opening for pretend vacations and flights to nowhere',
  'original URL': 'https://www.travelandleisure.com/airlines-airports/taipei-songshan-airport-taiwan-tour'},
 '2021-04-15': {'headline': 'The ‘Leave Britney Alone’ video was sold as an NFT for over $40,000',
  'original URL': 'https://foundation.app/ChrisCrocker/leave-britney-alone-21947'},
 '2021-05-20': {'headline': "High traces of Viagra in Seoul's sewers, research shows” ",
  'original URL': 'https://www.channelnewsasia.com/news/asia/high-traces-of-viagra-in-seoul-s-sewers--re

In [50]:
async def replace_article(date, headline, original_url):
    replace = await Runner.run(replacer, f"{date}, {headline}, {original_url}")
    return replace.final_output

In [52]:
for date, article in article_replacements.items():
    headline = article["headline"]
    original_url = article["original URL"]
    replacements = await replace_article(date, headline, original_url)
    print(f"Original headline: {headline}")
    print("Suggested replacements:")
    for result in replacements.results:
        print(result.title)
        print(result.url)
    print("\n")

Original headline: Need a loo fast? Book a ‘mobile toilet’ trailer on your smartphone
Suggested replacements:
Failed to search web
https://www.asahi.com/ajw/articles/AJ202001260004.html


Original headline: Italian designer creates the 'trikini,' a bikini set with matching face mask
Suggested replacements:
Reuters — Italian designer creates 'trikini' bikini with matching face mask
https://www.reuters.com/article/us-health-coronavirus-italy-fashion-idUSKBN22S1W0
ABC7 Chicago — Italian designer creates 'trikini' bikini with matching face mask
https://abc7chicago.com/fashion/italian-designer-creates-trikini-bikini-with-matching-face-mask/6205020/
The New Arab — Italian designer creates 'trikini' bikini with matching face mask
https://www.newarab.com/news/italian-designer-creates-trikini-bikini-matching-face-mask


Original headline: This airport in Taiwan is opening for pretend vacations and flights to nowhere
Suggested replacements:
Search unavailable
https://www.travelandleisure.com/air